# treefi Sample Usage

This notebook shows how to use `treefi` with:

- scikit-learn regression
- scikit-learn classification
- XGBoost sklearn-compatible wrappers
- CatBoost sklearn-compatible wrappers

The examples focus on `feature_interactions(...)`, `feature_importance(...)`, and `summarize_model(...)`.

In [ ]:
import treefi

from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import xgboost as xgb

## Data Setup

In [ ]:
diabetes = load_diabetes(as_frame=True)
X_reg = diabetes.frame[diabetes.feature_names]
y_reg = diabetes.frame[diabetes.target.name]

cancer = load_breast_cancer(as_frame=True)
X_clf = cancer.frame[cancer.feature_names]
y_clf = cancer.frame[cancer.target.name]

X_reg.head()

## 1. scikit-learn Regression Example

In [ ]:
rf_reg = RandomForestRegressor(n_estimators=10, max_depth=4, random_state=0)
rf_reg.fit(X_reg, y_reg)

reg_interactions = treefi.feature_interactions(
    rf_reg,
    max_interaction_depth=1,
    sort_by="gain",
    top_k=10,
)
reg_interactions.head()

In [ ]:
reg_importance = treefi.feature_importance(rf_reg, sort_by="gain", top_k=10)
reg_importance.head()

## 2. scikit-learn Classification Example

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=10, max_depth=4, random_state=0)
rf_clf.fit(X_clf, y_clf)

clf_interactions = treefi.feature_interactions(
    rf_clf,
    max_interaction_depth=1,
    sort_by="gain",
    top_k=10,
)
clf_interactions.head()

## 3. XGBoost sklearn API Examples

In [ ]:
xgb_reg = xgb.XGBRegressor(
    objective="reg:squarederror",
    max_depth=3,
    n_estimators=10,
    learning_rate=0.1,
    random_state=0,
)
xgb_reg.fit(X_reg, y_reg)

xgb_reg_interactions = treefi.feature_interactions(
    xgb_reg,
    max_interaction_depth=1,
    sort_by="gain",
    top_k=10,
)
xgb_reg_interactions.head()

In [ ]:
xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    max_depth=3,
    n_estimators=10,
    learning_rate=0.1,
    random_state=0,
)
xgb_clf.fit(X_clf, y_clf)

xgb_clf_interactions = treefi.feature_interactions(
    xgb_clf,
    max_interaction_depth=1,
    sort_by="gain",
    top_k=10,
)
xgb_clf_interactions.head()

## 4. CatBoost sklearn API Examples

In [ ]:
cat_reg = CatBoostRegressor(
    depth=3,
    iterations=10,
    learning_rate=0.1,
    verbose=False,
)
cat_reg.fit(X_reg, y_reg)

cat_reg_interactions = treefi.feature_interactions(
    cat_reg,
    max_interaction_depth=1,
    sort_by="interaction",
    top_k=10,
)
cat_reg_interactions.head()

In [ ]:
cat_clf = CatBoostClassifier(
    depth=3,
    iterations=10,
    learning_rate=0.1,
    verbose=False,
)
cat_clf.fit(X_clf, y_clf)

cat_clf_interactions = treefi.feature_interactions(
    cat_clf,
    max_interaction_depth=1,
    sort_by="interaction",
    top_k=10,
)
cat_clf_interactions.head()

## 5. Grouped Output with `summarize_model(...)`

In [ ]:
summary = treefi.summarize_model(xgb_reg, max_interaction_depth=1, top_k=10)

summary.metadata

In [ ]:
summary.interactions.head()

In [ ]:
summary.importance.head()

In [ ]:
summary.leaf_stats.head() if summary.leaf_stats is not None else None

## 6. Cross-Validated Stability Example

The CV helpers let you inspect whether strong interactions or features are stable across folds instead of trusting a single fitted model.

In [ ]:
cv_interactions = treefi.cross_validated_interactions(
    rf_reg,
    X_reg,
    y_reg,
    n_splits=5,
    top_k=10,
)

cv_interactions.metadata

In [ ]:
cv_interactions.interaction_summary[
    [
        "interaction",
        "mean_gain",
        "fold_presence_rate",
        "selection_rate_top_k",
        "rank_stability_score",
        "rare_fold_flag",
        "overfit_suspect_flag",
    ]
].sort_values("mean_gain", ascending=False).head(10)

A practical reading pattern is:

- look for high `mean_gain`
- prefer high `fold_presence_rate` and `selection_rate_top_k`
- be cautious when `rare_fold_flag` or `overfit_suspect_flag` is `True`

In [ ]:
cv_importance = treefi.cross_validated_importance(
    rf_reg,
    X_reg,
    y_reg,
    n_splits=5,
    top_k=10,
)

cv_importance.importance_summary[
    [
        "feature",
        "mean_gain",
        "mean_expected_gain",
        "fold_presence_rate",
        "selection_rate_top_k",
        "gain_cv",
    ]
].sort_values("mean_gain", ascending=False).head(10)